# Hestia A6 — AI Household Brain Dump

**Route A: Prompt Engineering · Person 1: Notebook + Research**

Hestia converts one household “brain dump” into separate, reviewable tasks. Each generated task must use the existing Django `Task` fields:

- `title`
- `task_type`
- `priority`
- `description`

This notebook defines the shared benchmark contract for all 15 models and evaluates Person 1's four lightweight models. It does **not** save generated tasks automatically; users must review and edit them before saving through Hestia.


## Reproducibility rules

Every teammate must use:

1. The same two Brain Dump inputs.
2. The same system instruction and JSON schema.
3. The same zero-, one-, few-, and many-shot examples.
4. Deterministic generation (`do_sample=False`) with at most 220 new tokens.
5. The scoring rubric below.

Run locally in a Python 3.11 environment. Model weights remain in the Hugging Face cache and must **not** be committed.


In [1]:
# Run only if the environment is missing these packages.
# %pip install "torch>=2.2" "transformers>=4.46" pandas nbformat

import gc
import json
import platform
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu",
})

{'python': '3.11.16', 'torch': '2.10.0', 'transformers': '5.2.0', 'device': 'mps'}


## Five model-size categories and final 15-model roster

Parameter boundaries are fixed before testing. Model assignments can be redistributed, but model IDs and categories should not change after benchmarking begins.


In [2]:
MODEL_ROSTER = [
    # Category 1: Tiny (up to 0.5B)
    {"category": "1 — Tiny (≤0.5B)", "model_id": "distilbert/distilgpt2", "params_b": 0.082, "owner": "P1"},
    {"category": "1 — Tiny (≤0.5B)", "model_id": "HuggingFaceTB/SmolLM2-135M-Instruct", "params_b": 0.135, "owner": "P1"},
    {"category": "1 — Tiny (≤0.5B)", "model_id": "HuggingFaceTB/SmolLM2-360M-Instruct", "params_b": 0.360, "owner": "P1"},
    {"category": "1 — Tiny (≤0.5B)", "model_id": "Qwen/Qwen2.5-0.5B-Instruct", "params_b": 0.494, "owner": "P1"},

    # Category 2: Small (over 0.5B through 2B)
    {"category": "2 — Small (>0.5–2B)", "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "params_b": 1.10, "owner": "P2"},
    {"category": "2 — Small (>0.5–2B)", "model_id": "Qwen/Qwen2.5-1.5B-Instruct", "params_b": 1.54, "owner": "P2"},
    {"category": "2 — Small (>0.5–2B)", "model_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "params_b": 1.70, "owner": "P2"},

    # Category 3: Medium (over 2B through 4B)
    {"category": "3 — Medium (>2–4B)", "model_id": "Qwen/Qwen2.5-3B-Instruct", "params_b": 3.09, "owner": "P2"},
    {"category": "3 — Medium (>2–4B)", "model_id": "microsoft/Phi-3.5-mini-instruct", "params_b": 3.82, "owner": "P3"},
    {"category": "3 — Medium (>2–4B)", "model_id": "meta-llama/Llama-3.2-3B-Instruct", "params_b": 3.21, "owner": "P3", "access": "gated"},

    # Category 4: Large (over 4B through 10B)
    {"category": "4 — Large (>4–10B)", "model_id": "Qwen/Qwen2.5-7B-Instruct", "params_b": 7.62, "owner": "P3"},
    {"category": "4 — Large (>4–10B)", "model_id": "mistralai/Mistral-7B-Instruct-v0.3", "params_b": 7.25, "owner": "P3"},
    {"category": "4 — Large (>4–10B)", "model_id": "tiiuae/Falcon3-7B-Instruct", "params_b": 7.00, "owner": "P4"},

    # Category 5: Very large (over 10B)
    {"category": "5 — Very large (>10B)", "model_id": "Qwen/Qwen2.5-14B-Instruct", "params_b": 14.7, "owner": "P4"},
    {"category": "5 — Very large (>10B)", "model_id": "microsoft/Phi-3-medium-4k-instruct", "params_b": 14.0, "owner": "P4"},
]

roster_df = pd.DataFrame(MODEL_ROSTER)
assert len(roster_df) == 15
assert roster_df["category"].nunique() == 5
roster_df

,category,model_id,params_b,owner,access
0,1 — Tiny (≤0.5B),distilbert/distilgpt2,0.082,P1,NaN
1,1 — Tiny (≤0.5B),HuggingFaceTB/SmolLM2-135M-Instruct,0.135,P1,NaN
2,1 — Tiny (≤0.5B),HuggingFaceTB/SmolLM2-360M-Instruct,0.360,P1,NaN
3,1 — Tiny (≤0.5B),Qwen/Qwen2.5-0.5B-Instruct,0.494,P1,NaN
4,2 — Small (>0.5–2B),TinyLlama/TinyLlama-1.1B-Chat-v1.0,1.100,P2,NaN
5,2 — Small (>0.5–2B),Qwen/Qwen2.5-1.5B-Instruct,1.540,P2,NaN
6,2 — Small (>0.5–2B),HuggingFaceTB/SmolLM2-1.7B-Instruct,1.700,P2,NaN
7,3 — Medium (>2–4B),Qwen/Qwen2.5-3B-Instruct,3.090,P2,NaN
8,3 — Medium (>2–4B),microsoft/Phi-3.5-mini-instruct,3.820,P3,NaN
9,3 — Medium (>2–4B),meta-llama/Llama-3.2-3B-Instruct,3.210,P3,gated


## Shared Brain Dump contract

Allowed values match Hestia’s existing `Task.TaskType` and `Task.Priority` choices exactly. Models must return JSON only. The two fixed inputs test both direct extraction and judgment about urgency.


In [3]:
TASK_TYPES = {"CHORE", "APPOINTMENT", "DEADLINE", "ERRAND", "MAINTENANCE", "SHOPPING", "OTHER"}
PRIORITIES = {"LOW", "MEDIUM", "HIGH", "URGENT"}
REQUIRED_FIELDS = {"title", "task_type", "priority", "description"}

SYSTEM_INSTRUCTION = '''You are Hestia's household task extraction assistant.
Convert the user's message into separate household tasks.
Return JSON only in this exact shape:
{"tasks":[{"title":"...","task_type":"CHORE|APPOINTMENT|DEADLINE|ERRAND|MAINTENANCE|SHOPPING|OTHER","priority":"LOW|MEDIUM|HIGH|URGENT","description":"..."}]}
Do not invent concerns that the user did not mention. Use URGENT only for an immediate safety risk or a deadline within 24 hours. Keep titles and descriptions concise.'''

TEST_INPUTS = {
    "BD1": "Guests are coming tomorrow. The kitchen is messy, we need groceries, and the bathroom tap is leaking badly.",
    "BD2": "This week I need to book the dentist, replace the smoke detector batteries, return a library book by Friday, and wash the guest-room sheets.",
}

EXAMPLES = [
    {
        "input": "The trash is full and we are out of milk.",
        "output": {"tasks": [
            {"title": "Take out the trash", "task_type": "CHORE", "priority": "MEDIUM", "description": "Empty the full household trash bin."},
            {"title": "Buy milk", "task_type": "SHOPPING", "priority": "MEDIUM", "description": "Purchase milk for the household."},
        ]},
    },
    {
        "input": "The gas stove smells like gas, and the hallway light bulb needs replacing.",
        "output": {"tasks": [
            {"title": "Check gas leak", "task_type": "MAINTENANCE", "priority": "URGENT", "description": "Stop using the stove and arrange an immediate gas safety inspection."},
            {"title": "Replace hallway light bulb", "task_type": "MAINTENANCE", "priority": "LOW", "description": "Replace the burned-out hallway light bulb."},
        ]},
    },
    {
        "input": "The dog needs a vet appointment and the living room needs vacuuming.",
        "output": {"tasks": [
            {"title": "Book vet appointment", "task_type": "APPOINTMENT", "priority": "MEDIUM", "description": "Schedule a veterinary appointment for the dog."},
            {"title": "Vacuum living room", "task_type": "CHORE", "priority": "LOW", "description": "Vacuum the living-room floor."},
        ]},
    },
]

In [4]:
def build_user_prompt(text, shot_mode):
    shot_counts = {"zero": 0, "one": 1, "few": 2, "many": 3}
    if shot_mode not in shot_counts:
        raise ValueError(f"Unknown shot mode: {shot_mode}")
    examples = EXAMPLES[:shot_counts[shot_mode]]
    example_text = "\n\n".join(
        f"Example input: {item['input']}\nExample output: {json.dumps(item['output'])}"
        for item in examples
    )
    prefix = f"{example_text}\n\n" if example_text else ""
    return f"{prefix}Household message: {text}\nJSON output:"


def extract_json(text):
    # Extract the first complete JSON object without accepting Python literals.
    def reject_duplicate_keys(pairs):
        value = {}
        for key, item in pairs:
            if key in value:
                raise ValueError(f"Duplicate JSON key: {key}")
            value[key] = item
        return value

    decoder = json.JSONDecoder(object_pairs_hook=reject_duplicate_keys)
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text[match.start():])
            return value
        except json.JSONDecodeError:
            continue
    raise ValueError("No valid JSON object found")


def validate_payload(payload):
    errors = []
    if not isinstance(payload, dict) or not isinstance(payload.get("tasks"), list):
        return ["Root must be an object containing a tasks list"]
    if not payload["tasks"]:
        errors.append("tasks must not be empty")
    for index, task in enumerate(payload["tasks"], start=1):
        if not isinstance(task, dict):
            errors.append(f"Task {index} is not an object")
            continue
        missing = REQUIRED_FIELDS - task.keys()
        if missing:
            errors.append(f"Task {index} missing: {sorted(missing)}")
        if task.get("task_type") not in TASK_TYPES:
            errors.append(f"Task {index} has invalid task_type")
        if task.get("priority") not in PRIORITIES:
            errors.append(f"Task {index} has invalid priority")
        if not str(task.get("title", "")).strip():
            errors.append(f"Task {index} has an empty title")
        if not str(task.get("description", "")).strip():
            errors.append(f"Task {index} has an empty description")
    return errors


# Small self-check for the shared parser and schema.
assert validate_payload(EXAMPLES[0]["output"]) == []
assert extract_json(f"result: {json.dumps(EXAMPLES[0]['output'])}") == EXAMPLES[0]["output"]

## Benchmark runner

Timing separates model loading from generation. `schema_valid` is automatic; content quality is scored manually after reading the complete output. Each model is unloaded before the next one to limit memory use.


In [5]:
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE in {"mps", "cuda"} else torch.float32
SHOT_MODES = ["zero", "one", "few", "many"]
PERSON_1_MODELS = [row["model_id"] for row in MODEL_ROSTER if row["owner"] == "P1"]


def format_for_model(tokenizer, user_prompt):
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"Instruction:\n{SYSTEM_INSTRUCTION}\n\nUser:\n{user_prompt}\n\nAssistant:\n"


def benchmark_model(model_id):
    started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=DTYPE)
    model.to(DEVICE)
    model.eval()
    load_seconds = time.perf_counter() - started
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    rows = []
    for input_id, brain_dump in TEST_INPUTS.items():
        for shot_mode in SHOT_MODES:
            prompt = format_for_model(tokenizer, build_user_prompt(brain_dump, shot_mode))
            encoded = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            generation_started = time.perf_counter()
            with torch.inference_mode():
                generated = model.generate(
                    **encoded,
                    max_new_tokens=220,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )
            generation_seconds = time.perf_counter() - generation_started
            new_tokens = generated[0, encoded["input_ids"].shape[1]:]
            raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            try:
                payload = extract_json(raw_output)
                validation_errors = validate_payload(payload)
            except (ValueError, json.JSONDecodeError) as exc:
                payload = None
                validation_errors = [str(exc)]
            rows.append({
                "model_id": model_id,
                "input_id": input_id,
                "shot_mode": shot_mode,
                "load_seconds": round(load_seconds, 3),
                "generation_seconds": round(generation_seconds, 3),
                "output_tokens": int(new_tokens.shape[-1]),
                "schema_valid": not validation_errors,
                "validation_errors": " | ".join(validation_errors),
                "raw_output": raw_output,
                "quality_score_0_to_10": None,
                "review_notes": "",
            })

    del model, tokenizer
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()
    return rows


print("Device:", DEVICE)
print("Person 1 models:", PERSON_1_MODELS)

Device: mps
Person 1 models: ['distilbert/distilgpt2', 'HuggingFaceTB/SmolLM2-135M-Instruct', 'HuggingFaceTB/SmolLM2-360M-Instruct', 'Qwen/Qwen2.5-0.5B-Instruct']


In [6]:
# This cell downloads public model weights on first run and may take several minutes.
all_rows = []
for model_id in PERSON_1_MODELS:
    print(f"Benchmarking {model_id}...")
    try:
        all_rows.extend(benchmark_model(model_id))
    except Exception as exc:
        all_rows.append({
            "model_id": model_id,
            "input_id": "MODEL_LOAD",
            "shot_mode": "n/a",
            "load_seconds": None,
            "generation_seconds": None,
            "output_tokens": None,
            "schema_valid": False,
            "validation_errors": f"{type(exc).__name__}: {exc}",
            "raw_output": "",
            "quality_score_0_to_10": None,
            "review_notes": "Model could not be benchmarked; retain the error as evidence.",
        })

results_df = pd.DataFrame(all_rows)
results_df

Benchmarking distilbert/distilgpt2...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Benchmarking HuggingFaceTB/SmolLM2-135M-Instruct...


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Benchmarking HuggingFaceTB/SmolLM2-360M-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Benchmarking Qwen/Qwen2.5-0.5B-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,model_id,input_id,shot_mode,load_seconds,generation_seconds,output_tokens,schema_valid,validation_errors,raw_output,quality_score_0_to_10,review_notes
0,distilbert/distilgpt2,BD1,zero,0.942,1.810,220,False,No valid JSON object found,Household message: Guests are coming tomorrow....,None,
1,distilbert/distilgpt2,BD1,one,0.942,1.510,220,True,,"Example input: ""Buy milk"", ""task_type"": ""SHOPP...",None,
2,distilbert/distilgpt2,BD1,few,0.942,1.619,220,True,,"Example input: The kitchen is messy, we need g...",None,
3,distilbert/distilgpt2,BD1,many,0.942,1.777,220,True,,Example input: The kitchen is clean and we are...,None,
4,distilbert/distilgpt2,BD2,zero,0.942,1.381,220,False,No valid JSON object found,Household message: This week I need to book th...,None,
5,distilbert/distilgpt2,BD2,one,0.942,1.959,220,True,,Example input: The trash is full and we are ou...,None,
6,distilbert/distilgpt2,BD2,few,0.942,1.420,220,True,,Example input: The trash is full and we are ou...,None,
7,distilbert/distilgpt2,BD2,many,0.942,1.464,220,True,,Example input: The dog needs a vet appointment...,None,
8,HuggingFaceTB/SmolLM2-135M-Instruct,BD1,zero,0.862,0.859,62,False,Task 1 has invalid task_type | Task 1 has inva...,"{""tasks"":[{""title"":""..."",""task_type"":""CHORE|AP...",None,
9,HuggingFaceTB/SmolLM2-135M-Instruct,BD1,one,0.862,3.545,220,False,No valid JSON object found,"{""tasks"":[{""title"":""Take out the trash"", ""task...",None,


## Human evaluation rubric (10 points)

Score every successful response using the same rubric:

| Criterion | Points | What to check |
|---|---:|---|
| Task extraction | 0–3 | All stated concerns become separate tasks; no invented tasks |
| Task type | 0–2 | Existing Hestia task types are appropriate |
| Priority | 0–2 | Urgency and deadlines are interpreted reasonably |
| Clarity | 0–2 | Titles and descriptions are concise and actionable |
| Format reliability | 0–1 | Valid JSON with the exact required fields |

Enter the total and a short justification in `quality_score_0_to_10` and `review_notes`. Do not invent scores for outputs that were not reviewed.


In [7]:
# Person 1 reviewed every raw output against the rubric above.
# Keys are (model_id, input_id, shot_mode); values are (score, concise justification).
MANUAL_REVIEWS = {
    ("distilbert/distilgpt2", "BD1", "zero"): (0, "Repeated the prompt; no tasks or JSON."),
    ("distilbert/distilgpt2", "BD1", "one"): (1, "Copied an example instead of extracting the user's concerns."),
    ("distilbert/distilgpt2", "BD1", "few"): (1, "Repeated examples; the parsed task is unrelated to the input."),
    ("distilbert/distilgpt2", "BD1", "many"): (1, "Copied the vet example and missed all requested tasks."),
    ("distilbert/distilgpt2", "BD2", "zero"): (0, "Repeated the prompt; no tasks or JSON."),
    ("distilbert/distilgpt2", "BD2", "one"): (1, "Copied the milk example and missed the four concerns."),
    ("distilbert/distilgpt2", "BD2", "few"): (1, "Repeated examples rather than producing input-specific tasks."),
    ("distilbert/distilgpt2", "BD2", "many"): (1, "Copied the vet example and missed the four concerns."),

    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD1", "zero"): (0, "Returned the schema placeholder, not extracted tasks."),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD1", "one"): (0, "Repeated one example until the output was truncated."),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD1", "few"): (0, "Repeated example fields and produced incomplete JSON."),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD1", "many"): (1, "Mentioned guests but missed the three actionable concerns."),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD2", "zero"): (0, "Returned the schema placeholder, not extracted tasks."),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD2", "one"): (2, "Recognized some concerns but collapsed them into duplicate JSON keys."),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD2", "few"): (2, "Mentioned the concerns but collapsed them into malformed JSON."),
    ("HuggingFaceTB/SmolLM2-135M-Instruct", "BD2", "many"): (3, "Started separate relevant tasks but repeated keys and was truncated."),

    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD1", "zero"): (7, "Extracted all three concerns clearly; task_type values were outside Hestia's choices."),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD1", "one"): (5, "Found groceries and the tap but replaced kitchen cleaning with the example trash task."),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD1", "few"): (7, "Extracted all concerns; grocery type and priorities need correction."),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD1", "many"): (6, "Extracted all concerns but overused MAINTENANCE and weak priorities."),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD2", "zero"): (3, "Recognized several concerns but used placeholder enums and incomplete JSON."),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD2", "one"): (7, "Extracted all four tasks clearly; most task types were incorrect."),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD2", "few"): (7, "Extracted all four tasks; types and deadline priority were weak."),
    ("HuggingFaceTB/SmolLM2-360M-Instruct", "BD2", "many"): (8, "Extracted four clear tasks; two classifications still need editing."),

    ("Qwen/Qwen2.5-0.5B-Instruct", "BD1", "zero"): (9, "Extracted all three concerns with useful priorities; groceries should be SHOPPING."),
    ("Qwen/Qwen2.5-0.5B-Instruct", "BD1", "one"): (6, "Handled groceries and the leak but invented dinner and switched descriptions to Chinese."),
    ("Qwen/Qwen2.5-0.5B-Instruct", "BD1", "few"): (5, "Invented dinner and mixed the grocery and tap descriptions."),
    ("Qwen/Qwen2.5-0.5B-Instruct", "BD1", "many"): (5, "Missed kitchen cleaning and mixed grocery and maintenance details."),
    ("Qwen/Qwen2.5-0.5B-Instruct", "BD2", "zero"): (7, "Extracted all four tasks; several types and the sheet priority were incorrect."),
    ("Qwen/Qwen2.5-0.5B-Instruct", "BD2", "one"): (6, "Extracted four tasks but invented details and ignored allowed task types."),
    ("Qwen/Qwen2.5-0.5B-Instruct", "BD2", "few"): (7, "Extracted four clear tasks; most types need correction."),
    ("Qwen/Qwen2.5-0.5B-Instruct", "BD2", "many"): (8, "Produced four usable tasks; invented a dentist/date detail and understated Friday urgency."),
}

for index, row in results_df.iterrows():
    key = (row["model_id"], row["input_id"], row["shot_mode"])
    if key in MANUAL_REVIEWS:
        results_df.at[index, "quality_score_0_to_10"] = MANUAL_REVIEWS[key][0]
        results_df.at[index, "review_notes"] = MANUAL_REVIEWS[key][1]

assert results_df["quality_score_0_to_10"].notna().all(), "Every successful P1 run must be reviewed"
results_df.groupby("model_id").agg(
    mean_quality=("quality_score_0_to_10", "mean"),
    json_success_rate=("schema_valid", "mean"),
    mean_generation_seconds=("generation_seconds", "mean"),
).round(3)

,mean_quality,json_success_rate,mean_generation_seconds
model_id,,,
HuggingFaceTB/SmolLM2-135M-Instruct,1.0,0.000,2.465
HuggingFaceTB/SmolLM2-360M-Instruct,6.25,0.750,3.148
Qwen/Qwen2.5-0.5B-Instruct,6.625,0.875,2.876
distilbert/distilgpt2,0.75,0.750,1.618


In [8]:
# Example review workflow: filter one model/input, read raw_output, then assign honest scores.
results_df[["model_id", "input_id", "shot_mode", "schema_valid", "quality_score_0_to_10", "raw_output"]]

,model_id,input_id,shot_mode,schema_valid,quality_score_0_to_10,raw_output
0,distilbert/distilgpt2,BD1,zero,False,0,Household message: Guests are coming tomorrow....
1,distilbert/distilgpt2,BD1,one,True,1,"Example input: ""Buy milk"", ""task_type"": ""SHOPP..."
2,distilbert/distilgpt2,BD1,few,True,1,"Example input: The kitchen is messy, we need g..."
3,distilbert/distilgpt2,BD1,many,True,1,Example input: The kitchen is clean and we are...
4,distilbert/distilgpt2,BD2,zero,False,0,Household message: This week I need to book th...
5,distilbert/distilgpt2,BD2,one,True,1,Example input: The trash is full and we are ou...
6,distilbert/distilgpt2,BD2,few,True,1,Example input: The trash is full and we are ou...
7,distilbert/distilgpt2,BD2,many,True,1,Example input: The dog needs a vet appointment...
8,HuggingFaceTB/SmolLM2-135M-Instruct,BD1,zero,False,0,"{""tasks"":[{""title"":""..."",""task_type"":""CHORE|AP..."
9,HuggingFaceTB/SmolLM2-135M-Instruct,BD1,one,False,0,"{""tasks"":[{""title"":""Take out the trash"", ""task..."


In [9]:
# Save portable results for P1's final merge. Model weights are never written to the repository.
RESULTS_DIR = Path("a6_results")
RESULTS_DIR.mkdir(exist_ok=True)
output_path = RESULTS_DIR / "person1_benchmark_results.csv"
results_df.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Generated at:", datetime.now(timezone.utc).isoformat())

Saved: a6_results/person1_benchmark_results.csv
Generated at: 2026-10-18T00:01:32.718977+00:00


## Team result handoff format

P2–P4 should submit one CSV with these exact columns:

`model_id, input_id, shot_mode, load_seconds, generation_seconds, output_tokens, schema_valid, validation_errors, raw_output, quality_score_0_to_10, review_notes`

Place their files in `a6_results/`. The merge cell below combines them without teammates editing this notebook.


In [10]:
RESULT_COLUMNS = [
    "model_id", "input_id", "shot_mode", "load_seconds", "generation_seconds",
    "output_tokens", "schema_valid", "validation_errors", "raw_output",
    "quality_score_0_to_10", "review_notes",
]

result_files = sorted(RESULTS_DIR.glob("person*_benchmark_results.csv"))
combined_df = pd.concat((pd.read_csv(path) for path in result_files), ignore_index=True) if result_files else pd.DataFrame(columns=RESULT_COLUMNS)
missing_columns = set(RESULT_COLUMNS) - set(combined_df.columns)
assert not missing_columns, f"Missing result columns: {sorted(missing_columns)}"
print(f"Combined {len(combined_df)} rows from {len(result_files)} file(s)")
combined_df.head()

Combined 32 rows from 1 file(s)


,model_id,input_id,shot_mode,load_seconds,generation_seconds,output_tokens,schema_valid,validation_errors,raw_output,quality_score_0_to_10,review_notes
0,distilbert/distilgpt2,BD1,zero,0.942,1.810,220,False,No valid JSON object found,Household message: Guests are coming tomorrow....,0,Repeated the prompt; no tasks or JSON.
1,distilbert/distilgpt2,BD1,one,0.942,1.510,220,True,NaN,"Example input: ""Buy milk"", ""task_type"": ""SHOPP...",1,Copied an example instead of extracting the us...
2,distilbert/distilgpt2,BD1,few,0.942,1.619,220,True,NaN,"Example input: The kitchen is messy, we need g...",1,Repeated examples; the parsed task is unrelate...
3,distilbert/distilgpt2,BD1,many,0.942,1.777,220,True,NaN,Example input: The kitchen is clean and we are...,1,Copied the vet example and missed all requeste...
4,distilbert/distilgpt2,BD2,zero,0.942,1.381,220,False,No valid JSON object found,Household message: This week I need to book th...,0,Repeated the prompt; no tasks or JSON.


## Part 3 Q1 — Size vs Quality

**Question:** How does model size affect quality for Hestia’s Household Brain Dump?

### Person 1 preliminary finding (four lightweight models, 32 runs)

| Model | Parameters | Mean quality /10 | Valid JSON | Mean generation time |
|---|---:|---:|---:|---:|
| DistilGPT2 | 82M | 0.750 | 6/8* | 1.534 s |
| SmolLM2 Instruct | 135M | 1.000 | 0/8 | 2.574 s |
| SmolLM2 Instruct | 360M | 6.250 | 6/8 | 3.240 s |
| Qwen2.5 Instruct | 494M | 6.625 | 7/8 | 2.896 s |

Quality generally increased with parameter count in this lightweight sample. The clearest same-family comparison was SmolLM2: moving from 135M to 360M increased mean reviewed quality from **1.0 to 6.25** and valid JSON from **0% to 75%**, with mean generation time increasing from **2.574 s to 3.240 s**. Qwen2.5-0.5B produced the highest mean quality (**6.625**) and format reliability (**87.5%**).

Size was not the only cause. DistilGPT2 is a base language model rather than an instruction-tuned chat model; it often copied examples instead of following the task. Its `6/8` parseable outputs therefore received only **0.75/10** on average. This shows why format validity must not be treated as task quality.

Across all four models, average quality rose from **3.25 zero-shot** to **4.125 many-shot**, but examples did not improve every individual model. Qwen’s zero-shot outputs were often better than its longer example-based prompts. The current recommendation is **Qwen2.5-0.5B-Instruct** among P1’s models because it gave the strongest quality/reliability balance without the largest generation time. The team should revise the final conclusion only after merging the remaining 11 models.

\*DistilGPT2’s “valid JSON” frequently came from copied demonstration examples, not correct input-specific tasks.


In [11]:
reviewed = combined_df.dropna(subset=["quality_score_0_to_10"]).copy()
if reviewed.empty:
    print("No human-reviewed scores yet. Review outputs before drawing a size-vs-quality conclusion.")
else:
    analysis_df = reviewed.merge(roster_df[["model_id", "category", "params_b"]], on="model_id", how="left")
    summary = (
        analysis_df.groupby(["category", "model_id", "params_b"], dropna=False)
        .agg(
            mean_quality=("quality_score_0_to_10", "mean"),
            json_success_rate=("schema_valid", "mean"),
            mean_generation_seconds=("generation_seconds", "mean"),
        )
        .reset_index()
        .sort_values("params_b")
    )
    display(summary)


,category,model_id,params_b,mean_quality,json_success_rate,mean_generation_seconds
3,1 — Tiny (≤0.5B),distilbert/distilgpt2,0.082,0.750,0.750,1.617500
0,1 — Tiny (≤0.5B),HuggingFaceTB/SmolLM2-135M-Instruct,0.135,1.000,0.000,2.465125
1,1 — Tiny (≤0.5B),HuggingFaceTB/SmolLM2-360M-Instruct,0.360,6.250,0.750,3.148375
2,1 — Tiny (≤0.5B),Qwen/Qwen2.5-0.5B-Instruct,0.494,6.625,0.875,2.875750
